# 06 — Multi-model comparison

Central notebook that **loads the forecasts generated by notebooks 01–05** and produces:

1. A unified metrics table (the 6) per series × model.
2. Paired Wilcoxon signed-rank between all models.
3. Month-by-month Diebold-Mariano per series.
4. Holm-Bonferroni correction per family.
5. Ranking of the best model per series.
6. Annual-usage aggregates consumed by the recommendation system (notebook 08).

**Environment:** main `.venv`.

## 1. Configuration

In [ ]:
from pathlib import Path
import sys
from itertools import combinations
import numpy as np
import pandas as pd

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.statistical_tests import wilcoxon_paired, diebold_mariano, holm_bonferroni
from src.metrics import all_metrics

OUTPUTS = REPO_ROOT / 'outputs'
COMPARISON_DIR = OUTPUTS / 'comparison'
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load each model's forecasts

Each previous notebook exports a CSV with columns `series_id, horizon_month, y_true, pred_<model>`. Here we join them.

In [ ]:
pred_paths = {
    'TimesFM_bare': OUTPUTS / 'timesfm_bare' / 'predictions_timesfm_bare.csv',
    'TimesFM_cov': OUTPUTS / 'timesfm_cov' / 'predictions_timesfm_cov.csv',
    'Chronos_Bolt': OUTPUTS / 'chronos' / 'predictions_chronos.csv',
    'TabPFN_TS': OUTPUTS / 'tabpfn' / 'predictions_tabpfn_ts.csv',
    'PyCaret_best': OUTPUTS / 'pycaret' / 'predictions_pycaret.csv',
    'Seasonal_Naive': OUTPUTS / 'baseline' / 'predictions_seasonal_naive.csv',
}

preds = None
for name, path in pred_paths.items():
    df_m = pd.read_csv(path)
    pred_col = [c for c in df_m.columns if c.startswith('pred_')][0]
    df_m = df_m.rename(columns={pred_col: name})
    base = df_m[['series_id', 'horizon_month', 'y_true', name]]
    if preds is None:
        preds = base
    else:
        preds = preds.merge(
            base.drop(columns='y_true'),
            on=['series_id', 'horizon_month'],
            how='inner',
        )
preds.head()

## 3. Load training series for the MASE denominator

In [ ]:
from src.data_loader import load_parquet, filter_period, build_series

df_raw = load_parquet(REPO_ROOT / 'data' / 'anonymized_series.parquet')
df_raw = filter_period(df_raw, 2020, 2024)
series = build_series(df_raw, min_months=24)
trains = {sid: s.values[:-12] for sid, s in series.items()}

## 4. Unified metrics table

In [ ]:
models = list(pred_paths.keys())
rows = []
for sid, group in preds.groupby('series_id'):
    group = group.sort_values('horizon_month')
    y_true = group['y_true'].values
    train = trains.get(sid)
    if train is None or len(train) < 24:
        continue
    for m in models:
        y_pred = group[m].values
        if np.any(pd.isna(y_pred)):
            continue
        met = all_metrics(y_true, y_pred, train)
        rows.append({'series_id': sid, 'model': m, **met})
metrics = pd.DataFrame(rows)
metrics.to_csv(COMPARISON_DIR / 'multimodel_comparison_metrics.csv', index=False)
metrics.groupby('model')[['MASE', 'MAE', 'RMSE', 'sMAPE', 'MedAE']].median().round(3)

## 5. Paired Wilcoxon between all models

In [ ]:
pivot_mase = metrics.pivot(index='series_id', columns='model', values='MASE').dropna()
wilcoxon_rows = []
for a, b in combinations(models, 2):
    if a not in pivot_mase.columns or b not in pivot_mase.columns:
        continue
    res = wilcoxon_paired(pivot_mase[a].values, pivot_mase[b].values, alternative='less')
    wilcoxon_rows.append({
        'model_A': a,
        'model_B': b,
        'n': res.n,
        'wins_A': res.wins_a,
        'wins_B': res.wins_b,
        'p_value': res.p_value,
        'effect_r': res.effect_size_r,
    })
wilcoxon_df = pd.DataFrame(wilcoxon_rows)

# Holm-Bonferroni correction
holm = holm_bonferroni(wilcoxon_df['p_value'].tolist(), alpha=0.05)
wilcoxon_df['holm_adjusted_p'] = holm['p_adjusted']
wilcoxon_df['reject_h0'] = holm['reject']
wilcoxon_df.to_csv(COMPARISON_DIR / 'wilcoxon_holm.csv', index=False)
wilcoxon_df

## 6. Month-by-month Diebold-Mariano per series

In [ ]:
dm_rows = []
for sid, group in preds.groupby('series_id'):
    group = group.sort_values('horizon_month')
    y_true = group['y_true'].values
    for a, b in combinations(models, 2):
        y_a = group[a].values
        y_b = group[b].values
        if np.any(pd.isna(y_a)) or np.any(pd.isna(y_b)):
            continue
        e_a = y_true - y_a
        e_b = y_true - y_b
        res = diebold_mariano(e_a, e_b, h=12, loss='mse')
        dm_rows.append({
            'series_id': sid,
            'model_A': a,
            'model_B': b,
            'dm_stat': res.dm_stat,
            'p_value': res.p_value,
        })
dm_df = pd.DataFrame(dm_rows)
dm_df.to_csv(COMPARISON_DIR / 'diebold_mariano_per_series.csv', index=False)

## 7. Best model per series

In [ ]:
best = metrics.loc[metrics.groupby('series_id')['MASE'].idxmin(), ['series_id', 'model', 'MASE']]
best.columns = ['series_id', 'best_model', 'best_MASE']
best.to_csv(COMPARISON_DIR / 'best_model_per_series.csv', index=False)
best['best_model'].value_counts(normalize=True).round(3)

## 8. Annual-usage aggregates for the recommendation system

The recommendation system (notebook 08) compares the **predicted annual usage**
(sum of the 12 forecast months, per model) against the **actual usage of the
last observed year** (sum of the 12 test months).

In [ ]:
# Actual usage of the last observed year (one row per series)
actual_usage_last = (
    preds.drop_duplicates(['series_id', 'horizon_month'])
    .groupby('series_id')['y_true'].sum()
    .rename('actual_usage').reset_index()
)
actual_usage_last.to_csv(COMPARISON_DIR / 'actual_usage_last_year.csv', index=False)

# Predicted annual usage per (series, model)
next_rows = []
for m in models:
    agg = preds.groupby('series_id')[m].sum()
    for sid, val in agg.items():
        if pd.notna(val):
            next_rows.append({'series_id': sid, 'model': m, 'predicted_usage': float(val)})
next_predictions = pd.DataFrame(next_rows)
next_predictions.to_csv(COMPARISON_DIR / 'next_predictions.csv', index=False)
next_predictions.head()